# Solutions — Performance

One solution per exercise and per mini challenge, in lesson order.
Read these **after** you have tried. A solution you have not attempted teaches nothing.

Playground answers are given as prose plus the code to paste, because they cannot run on this
kernel. The numbers quoted are the ones measured while writing the lessons — yours will differ in
value and should not differ in shape.

### LESSON 68 — Exercise

**1. A component with no props.** It re-renders. Reason 2 from the lesson — its parent rendered —
does not mention props at all. React does not compare anything before calling a child; a child
with no props is called exactly as often as one with ten.

```jsx
function Sibling() {
  countRender("Sibling");
  return <li>no props at all</li>;
}
```

**2. Moving the state down.**

```jsx
function Counter() {
  const [count, setCount] = useState(0);
  countRender("Counter");
  return (
    <button id="inc" onClick={() => setCount(count + 1)}>
      App state: {count}
    </button>
  );
}
```

Now pressing the button renders `Counter` and nothing else — `App` does not re-render, so none of
its other children do either. Measured: six components dropped to one. No `memo`, no `useMemo`,
no new API — the fix was **where the state lives** (LESSON 66), and it is by a wide margin the
most effective thing in this topic.

**3. Moving the list inside `Wrapper` as children.**

```jsx
<Wrapper>
  <ul>
    <PlainChild label="steady" />
    …
  </ul>
</Wrapper>
```

Pressing **wrapper state** does *not* re-render them: `Wrapper` re-renders, but `children` is the
same element object it received from `App`, and `App` did not run. Pressing **App state** *does*
re-render them, because `App` re-creates those elements.

The rule underneath: a component re-renders when the element describing it is created again. When
you pass JSX as `children`, the creating component is the parent, not the wrapper.

**4. The Profiler.** A commit in this experiment is around one millisecond. Nothing here is worth
optimising, and that is the intended answer — the experiment exists to make render *counts*
visible, and a visible count is not a problem by itself. If you added `memo` to everything on the
evidence of the console, you would have added code and changed nothing a user can feel.

### LESSON 68 — Mini challenge

In [ ]:
// L68 solution — a render-count budget

const l68Tree = { header: 1, sidebar: 12, rows: 200, perRow: 3, footer: 1 };
const l68Total =
  1 + l68Tree.header + l68Tree.sidebar + l68Tree.rows * l68Tree.perRow + l68Tree.footer;

function l68Renders(where) {
  if (where === "App") return l68Total;                       // the whole tree
  if (where === "sidebar") return l68Tree.sidebar;            // the sidebar's own subtree
  if (where === "one row") return l68Tree.perRow;             // that row and its children
  throw new Error(`unknown: ${where}`);
}

for (const where of ["App", "sidebar", "one row"]) {
  console.log(`state in ${where.padEnd(9)} -> ${l68Renders(where)} component calls`);
}

// The search box is in the header and the table shows the results, so both branches read it:
// its closest common parent is App, and every keystroke costs the "App" number — 615 calls.
//
// What that tells you: the fix cannot come from memo. memo would have to be applied to the
// sidebar, the footer and 200 rows to claw back what one placement decision cost, and every one
// of those is a comparison that also costs something. The real options are LESSON 66's — is the
// query really needed by both branches? — and, if it genuinely is, keeping the expensive branch
// out of the urgent path (LESSON 72).

**Common mistake:** counting DOM nodes instead of component calls, or forgetting that "state in
one row" still re-renders that row's children. The number React cares about is how many of *your
functions* it has to call.

### LESSON 69 — Exercise

In [ ]:
// L69 solution — which props failed the comparison

function l69sShallowEqual(previous, next) {
  const a = Object.keys(previous);
  const b = Object.keys(next);
  if (a.length !== b.length) return false;
  return a.every((key) => Object.is(previous[key], next[key]));
}

function l69sExplain(previous, next) {
  const keys = new Set([...Object.keys(previous), ...Object.keys(next)]);
  return [...keys].filter((key) => !Object.is(previous[key], next[key]));
}

const l69sRender1 = { label: "steady", config: { label: "x" }, onPing: () => {} };
const l69sRender2 = { label: "steady", config: { label: "x" }, onPing: () => {} };

console.log("equal?      ", l69sShallowEqual(l69sRender1, l69sRender2));
console.log("failed props:", l69sExplain(l69sRender1, l69sRender2));

**2. `label={`steady ${count}`}`.** `MemoPrimitive` now re-renders on every press: the string is
different each time, so the comparison fails honestly. The condition violated is React's
*"re-renders often with the same exact props"* — the props are not the same, so `memo` is
useless by construction. It is not broken; it is being asked to do something impossible, and you
are paying for the comparison.

**3. Adding an equal second prop.** It does not help at all. The comparison is an `every` over the
props: one failure is enough. `memo` would need **every** prop to pass, so the object prop must
keep the same reference between renders — which means it has to be created somewhere that does not
re-run each render. That is `useMemo`, and it is LESSON 70's subject.

**Common mistake:** concluding `memo` is broken. It did exactly what it promises — a shallow
comparison — and the promise is less useful than people assume.

### LESSON 69 — Mini challenge

In [ ]:
// L69 solution — the memo audit

const l69sSharedRows = [1, 2];
const l69sHandler = () => {};

const l69sCases = [
  ["id + name",        { id: 3, name: "Ada" },        { id: 3, name: "Ada" }],
  ["same array",       { rows: l69sSharedRows },      { rows: l69sSharedRows }],
  ["new array",        { rows: [1, 2] },              { rows: [1, 2] }],
  ["handler in state", { onSave: l69sHandler },       { onSave: l69sHandler }],
  ["extra prop",       { id: 3 },                     { id: 3, tone: undefined }],
];

for (const [name, before, after] of l69sCases) {
  const skips = l69sShallowEqual(before, after);
  console.log(`${name.padEnd(18)} memo skips? ${skips}`);
}

// The extra prop: memo does NOT skip. Object.keys({id:3}) has length 1 and
// Object.keys({id:3, tone: undefined}) has length 2, so the key-count check fails before any
// value is compared — even though `tone` is undefined and the JSX <Child id={3} tone={undefined} />
// looks, to a reader, exactly like <Child id={3} />. Conditionally spreading a prop
// ({...(dark && { tone: "dark" })}) is the usual way people create this by accident.
//
// The two shapes you can get without useMemo, purely by WHERE you declare the value:
//   - "same array": defined outside the component (module level) or held in state/a ref
//   - "handler in state": the same — a stable reference that render does not recreate
// Both are free. Reach for useMemo only when the value must be computed from props or state.

### LESSON 70 — Exercise

In [ ]:
// L70 solution — where does this calculation cross 1 ms?

const l70sMake = (n) =>
  Array.from({ length: n }, (_, i) => ({
    id: i,
    name: `item-${i}`,
    price: (i * 7919) % 1000,
    tag: ["a", "b", "c"][i % 3],
  }));

function l70sVisible(list, tag) {
  return list
    .filter((item) => item.tag === tag)
    .sort((a, b) => a.price - b.price || a.name.localeCompare(b.name))
    .slice(0, 20);
}

function l70sThreshold() {
  console.log("length     avg ms   verdict");
  for (const n of [200, 1000, 5000, 20000, 50000]) {
    const list = l70sMake(n);
    l70sVisible(list, "b");                    // one warm-up run, not measured
    let total = 0;
    for (let i = 0; i < 10; i += 1) {
      const start = performance.now();
      l70sVisible(list, "b");
      total += performance.now() - start;
    }
    const avg = total / 10;
    console.log(
      `${String(n).padEnd(10)} ${avg.toFixed(2).padStart(6)}   ${avg >= 1 ? "worth memoizing" : "not worth it"}`,
    );
  }
}

l70sThreshold();

// Which of those lengths does a screen actually hold? None of them. A table shows tens of rows,
// a page of results twenty. The lengths where this crosses 1 ms are lengths you would never
// render — which is why "filtering the list I am about to display" is almost never the
// calculation that needs useMemo. The list being long is a reason to render less of it
// (LESSON 73), not a reason to cache the filtering.

**2 and 3 — in the playground.**

```jsx
const config = useMemo(() => ({ label: "stable" }), []);
const onPing = useCallback(() => {}, []);
```

Measured before and after, pressing **App state**:

| component | before | after |
|---|---|---|
| `MemoObject` | 1 render per press | **0** |
| `MemoCallback` | 1 render per press | **0** |

Remove `memo` from `MemoObject` while keeping the `useMemo` and it re-renders on every press
again. The `useMemo` is then buying you **nothing**: no one is comparing the reference. That is
the direction to remember — `useMemo` for identity is only worth anything when something
downstream compares identity.

**3.** `useCallback` around a handler passed to a plain `<button>` is worth nothing. The
comparison that `useCallback` exists to satisfy is done by `memo` or by a dependency array; a DOM
element does no such comparison — React just attaches the newest function. You have added a Hook,
a dependency array and a line of noise to optimise a comparison nobody performs.

### LESSON 70 — Mini challenge

In [ ]:
// L70 solution — the stale dependency

function l70sCache(compute, deps) {
  const cache = { deps: null, value: undefined, hits: 0, misses: 0 };
  return function cached() {
    const fresh = deps();
    const same =
      cache.deps !== null &&
      cache.deps.length === fresh.length &&
      cache.deps.every((d, i) => Object.is(d, fresh[i]));
    if (same) {
      cache.hits += 1;
      return cache.value;
    }
    cache.misses += 1;
    cache.deps = fresh;
    cache.value = compute();
    return cache.value;
  };
}

const l70sItems = l70sMake(600);
let l70sTag = "a";

// the bug: `tag` is used by the calculation but missing from the dependency list
const l70sBuggy = l70sCache(() => l70sVisible(l70sItems, l70sTag), () => [l70sItems]);

console.log("first call  (tag = a):", l70sBuggy()[0]);
l70sTag = "b";
console.log("second call (tag = b):", l70sBuggy()[0]);

// and the same thing with the dependency declared honestly
const l70sFixed = l70sCache(() => l70sVisible(l70sItems, l70sTag), () => [l70sItems, l70sTag]);
l70sTag = "a";
console.log("\nfixed, tag = a:", l70sFixed()[0]);
l70sTag = "b";
console.log("fixed, tag = b:", l70sFixed()[0]);

// The second call returns the tag-"a" result: the deps did not change, so the cache answered.
// That is a CORRECTNESS bug, not a performance one — the screen shows data that does not match
// the filter, and it will be reported as "the filter doesn't work", not as "the app is slow".
//
// Which would I rather ship? The missing useMemo, every time. Its worst case is some wasted
// milliseconds that a Profiler can find. The missing dependency's worst case is wrong data
// displayed confidently, and nothing in the tooling flags it except the lint rule — which is
// the real reason to leave the exhaustive-deps rule switched on.

### LESSON 71 — Exercise

**1. A stuttering screen and no Profiler recording.** Turning on the compiler is a guess. It
inserts memoization, which helps only if the cost is *repeated work with unchanged inputs*. If the
stutter is one 300 ms calculation, or 5000 DOM nodes, or a synchronous call in an event handler,
the compiler changes nothing. Record first; the compiler is a fix for a diagnosis, not a substitute
for one.

**2. "We don't need to understand `memo` any more."** Right about the *writing*: on a compiled
codebase you should not be scattering `memo`, `useMemo` and `useCallback` by hand. Wrong about the
*understanding* — you still have to read a Profiler recording, and every question it raises is
phrased in exactly the vocabulary of lessons 68–70: why did this re-render, were the props the
same, is this calculation expensive. The compiler automates the typing, not the diagnosis.

**3. A component that mutates an array in state.** It breaks a Rule of React, and the compiler's
optimisation depends on those rules holding. Best case it declines to optimise the component; worse,
the mutation was already a latent bug (the same one LESSON 27 is about) that has been surviving only
because React re-rendered more often than it needed to. Fix the mutation — the compiler has done you
a favour by making it matter.

**4. A plain function scoring 6000 records, called from two components.** The compiler memoizes
components and Hooks, so it may cache each *component's* result — but React's docs are explicit
that it only memoizes React components and hooks, not every function, and that memoization is not
shared across components. The scoring function itself, and the fact that two components each pay
for it, stay your problem: cache it outside React, or compute it once higher up and pass the result
down.

### LESSON 71 — Mini challenge

In [ ]:
// L71 solution — the memoization React cannot do for you

const l71sRecords = Array.from({ length: 6000 }, (_, i) => ({ id: i, value: (i * 37) % 1000 }));

function l71sScore(records) {
  let sum = 0;
  for (const r of records) sum += r.value;
  const mean = sum / records.length;
  let variance = 0;
  for (const r of records) variance += (r.value - mean) ** 2;
  return { sum, mean: +mean.toFixed(2), sd: +Math.sqrt(variance / records.length).toFixed(2) };
}

function l71sTime(label, fn) {
  const start = performance.now();
  const value = fn();
  const ms = performance.now() - start;
  console.log(`${label.padEnd(34)} ${ms.toFixed(3)} ms`);
  return { value, ms };
}

console.log("without a cache:");
const l71sA = l71sTime("component A calls l71sScore", () => l71sScore(l71sRecords));
const l71sB = l71sTime("component B calls l71sScore", () => l71sScore(l71sRecords));

const l71sCache = new WeakMap();
function l71sMemoized(records) {
  if (l71sCache.has(records)) return l71sCache.get(records);
  const result = l71sScore(records);
  l71sCache.set(records, result);
  return result;
}

console.log("\nwith a module-level cache keyed by the array:");
const l71sC = l71sTime("component A calls l71sMemoized", () => l71sMemoized(l71sRecords));
const l71sD = l71sTime("component B calls l71sMemoized", () => l71sMemoized(l71sRecords));

console.log("\nsecond call, uncached vs cached:", l71sB.ms.toFixed(3), "ms vs", l71sD.ms.toFixed(3), "ms");
console.log("same object returned?", l71sC.value === l71sD.value);

// React's docs say memoization "is not shared across multiple components or hooks" — that is the
// limitation of the FIRST version, and of useMemo, and of the compiler: each component has its
// own cache, so component B pays in full. The WeakMap version is shared, because it lives in the
// module rather than in any component's Hook state.
//
// Why a plain cache rather than a Hook: the value depends only on its argument, not on any
// component's lifecycle, and it must be shared BETWEEN components. A Hook can be neither — it is
// per-component by construction. This is the same rule as LESSON 64's, arriving from the other
// direction: a Hook shares logic, never state. A WeakMap also lets the entry be collected once
// nothing else references the array, which a module-level Map would not.

### LESSON 72 — Exercise

**1. The blocking measurement.** Typing `person-1` one character at a time, measuring only the
time React blocked the typing loop: about **270 ms** in total with a plain `setQuery`, about
**6 ms** inside `startTransition`. Both end with the same list. If your numbers are smaller, your
machine is faster; the ratio is what matters.

**2. Removing `memo` from `Results`.** The Transition mode now renders the expensive list
**twice per keystroke** — 16 renders for 8 keystrokes, against 8 in urgent mode. LESSON 68's two
reasons explain it exactly: the urgent `setText` re-renders the parent, which re-renders `Results`
(reason 2 — its parent rendered) with the *old* query; then the Transition renders it again with
the new one. A Transition splits one update into two passes, and an unmemoized child does the work
in both.

**3. Putting `setText` inside the Transition.** The input stops keeping up: characters appear late,
in bursts, and fast typing can drop or reorder what you see, because the field's displayed value is
now allowed to lag behind the keys. React states the rule directly — Transitions cannot be used for
state that controls a text input, which must update synchronously. This is the one mistake in this
lesson that makes the app worse than it was before you optimised it.

**4. With `ROWS` reduced to 200.** The list render is now under a millisecond, the two modes are
indistinguishable, and `isPending` is never true long enough to see. The Transition is correct code
doing nothing — it costs a Hook, a callback and a reader's attention. Measured, not assumed: that is
the answer the topic wants.

### LESSON 72 — Mini challenge

In [ ]:
// L72 solution — three strategies on one timeline

function l72Timeline(keystrokes, workMs, strategy) {
  const gap = 50;
  const times = Array.from({ length: keystrokes }, (_, i) => i * gap);
  const runs = [];

  if (strategy === "none") {
    for (const t of times) runs.push({ startedAt: t, finishedAt: t + workMs, completed: true });
  }

  if (strategy === "debounce") {
    const last = times[times.length - 1];
    runs.push({ startedAt: last + 400, finishedAt: last + 400 + workMs, completed: true });
  }

  if (strategy === "transition") {
    times.forEach((t, i) => {
      const next = times[i + 1];
      const finishedAt = t + workMs;
      const abandoned = next !== undefined && next < finishedAt;
      runs.push({ startedAt: t, finishedAt: abandoned ? next : finishedAt, completed: !abandoned });
    });
  }

  const completed = runs.filter((r) => r.completed);
  return {
    strategy,
    started: runs.length,
    completed: completed.length,
    firstResultAt: completed.length ? completed[0].finishedAt : null,
    lastResultAt: completed.length ? completed[completed.length - 1].finishedAt : null,
    totalWorkMs: runs.reduce((sum, r) => sum + (r.finishedAt - r.startedAt), 0),
  };
}

for (const strategy of ["none", "debounce", "transition"]) {
  console.log(l72Timeline(8, 120, strategy));
}

// Soonest something on screen: "none" — it finishes the first render at 120 ms. It is also the
// strategy that blocks every keystroke to do it, which is exactly the trade the lesson measured.
// Least total work: "debounce" — one run, 120 ms, nothing wasted.
// "transition" starts every run but abandons the superseded ones; the work it does is real but it
// is interruptible, so it never makes the user wait.
//
// Why both, for a search box that hits the network: they solve different problems. Debouncing is
// about not sending eight REQUESTS — a network and server concern, and no amount of concurrency
// makes a wasted request free. Transitions are about not blocking the main thread while RENDERING
// what came back. Debounce the fetch, mark the rendering of the results as non-urgent.

### LESSON 73 — Exercise

**1. 50 rows against 5000.** Roughly 0.4–0.9 ms against about 48 ms to create the elements, plus
5000 DOM nodes the browser must lay out. Switching to `key={index}` and removing the first row:
with `key={row.id}` React moves the existing nodes and removes one; with `key={index}` every row
from the removal point on has a *different* key than before, so React updates the contents of all
of them. The list looks identical and the browser does far more work — and any state inside a row
(an open menu, a checkbox) stays with the index and jumps to a different item, which is LESSON 20's
original point.

**2. Slow 3G.** With the fallback on screen for a second or more, "Loading the chart…" is fine but
bare — a skeleton the size of the chart is better, because it prevents the layout jumping when the
real thing lands. That is LESSON 74's subject, one topic away.

**3. `<Suspense>` outside the condition.** Behaviour is unchanged: an idle `<Suspense>` with no
suspending child renders its children normally. Remove it entirely and React throws — the message
names the component and says a component suspended while responding to synchronous input, or that
no Suspense boundary was found; the fallback is not optional, it is how React knows what to show.

**4. A lazy component rendered on first paint.** It works, and it makes the app slower: you have
moved code out of the main bundle and then immediately asked for it, adding a network round trip to
the path the user is already waiting on. Splitting only pays for code the user might never request.

### LESSON 73 — Mini challenge

In [ ]:
// L73 solution — a splitting budget

const l73Features = [
  { name: "chart library",  kb: 95, note: "one screen of nine" },
  { name: "date picker",    kb: 30, note: "two forms" },
  { name: "markdown editor", kb: 60, note: "one session in twenty" },
  { name: "your own code",  kb: 35, note: "everywhere" },
];

const l73Total = l73Features.reduce((sum, f) => sum + f.kb, 0);

function l73Budget(features) {
  return features.map((f) => ({
    name: f.name,
    kb: f.kb,
    firstDownloadIfBundled: l73Total,
    firstDownloadIfSplit: l73Total - f.kb,
    saved: f.kb,
    note: f.note,
  }));
}

console.log(`total bundle: ${l73Total} kB\n`);
for (const row of l73Budget(l73Features)) {
  const decision =
    row.name === "your own code" ? "DO NOT SPLIT — needed on first paint"
    : row.name === "date picker" ? "probably not — 30 kB against a round trip in a form the user opened deliberately"
    : "SPLIT — large, and most sessions never load it";
  console.log(
    `${row.name.padEnd(16)} ${String(row.kb).padStart(3)} kB  ->  first download ${row.firstDownloadIfSplit} kB   ${decision}`,
  );
}

// The rule: split what is large AND unlikely to be needed in a given session. Both halves matter —
// size alone would have you splitting your own code, and unlikeliness alone would have you
// splitting a 2 kB dialog for no gain.
//
// The one the arithmetic gets wrong: the DATE PICKER. Splitting it "saves" 30 kB on paper, but it
// is used in two forms the user reaches deliberately, and the split turns a click into a click plus
// a network round trip — on a slow connection, a visible pause in the middle of a task. What the
// arithmetic leaves out is WHEN the cost is paid: bytes moved out of the first download are not
// deleted, they are rescheduled to a moment when the user is already waiting for something.